# Stock Data Collection and Feature Engineering

This notebook handles:
1. Data collection from Yahoo Finance
2. Feature engineering and technical indicators
3. Data preprocessing pipeline setup

In [1]:
# Import necessary libraries
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import ta  # Technical Analysis library

## Feature Engineering Functions

In [2]:
def add_technical_indicators(df):
    """Add technical indicators to the dataframe
    
    Args:
        df (pd.DataFrame): OHLCV dataframe
        
    Returns:
        pd.DataFrame: DataFrame with technical indicators
    """
    # Price-based indicators
    
    df['Returns'] = df['Close'].pct_change()
    df['MA5'] = df['Close'].rolling(window=5).mean()
    df['MA20'] = df['Close'].rolling(window=20).mean()
    
    # Volume indicators
    df['Volume_MA5'] = df['Volume'].rolling(window=5).mean()
    
    # Momentum indicators
    df['RSI'] = ta.momentum.RSIIndicator(df['Close']).rsi()
    
    # Volatility indicators
    df['ATR'] = ta.volatility.AverageTrueRange(df['High'], df['Low'], df['Close']).average_true_range()
    
    return df

## Data Pipeline

In [3]:
def create_stock_features(ticker_symbol, start_date, end_date=None):
    """Complete pipeline for fetching and processing stock data
    
    Args:
        ticker (str): Stock ticker symbol
        start_date (str): Start date in 'YYYY-MM-DD' format
        end_date (str, optional): End date in 'YYYY-MM-DD' format
        
    Returns:
        pd.DataFrame: Processed dataframe with all features
    """
    # Fetch raw data
    ticker_object = yf.Ticker(ticker_symbol)
    df = ticker_object.history(start=start_date, end=end_date)
    
    # Add technical indicators
    df = add_technical_indicators(df)
    
    # Clean data
    df = df.dropna()  # Remove any rows with missing values
    
    return df

## Example Usage

In [4]:
# Example: Fetch and process data for AAPL
ticker = '^GSPC'
start_date = '2010-01-01'
df = create_stock_features(ticker, start_date)
df.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits,Returns,MA5,MA20,Volume_MA5,RSI,ATR
Date,,,,,,,,,,,,,
2010-02-01 00:00:00-05:00,1073.890015,1089.380005,1073.890015,1089.189941,4077610000,0.0,0.0,0.014266,1087.452002,1121.862000,4.998778e+09,36.230451,15.496363
2010-02-02 00:00:00-05:00,1090.050049,1104.729980,1087.959961,1103.319946,4749540000,0.0,0.0,0.012973,1089.681982,1120.378497,5.002304e+09,43.998375,15.587338
2010-02-03 00:00:00-05:00,1100.670044,1102.719971,1093.969971,1097.280029,4285450000,0.0,0.0,-0.005474,1089.637988,1118.416498,4.795570e+09,41.662191,15.141812
2010-02-04 00:00:00-05:00,1097.250000,1097.250000,1062.780029,1063.109985,5859690000,0.0,0.0,-0.031141,1085.353979,1114.714996,4.877028e+09,31.478881,16.524540
2010-02-05 00:00:00-05:00,1064.119995,1067.130005,1044.500000,1066.189941,6438900000,0.0,0.0,0.002897,1083.817969,1110.939996,5.082238e+09,33.066956,16.960645


In [5]:
# Convert timestamps to same timezone
search_start_date = pd.Timestamp('2024-01-01').tz_localize(df.index.tz)
search_end_date = df.index.max()

# Get data slice
df.loc[search_start_date:search_end_date]


,Open,High,Low,Close,Volume,Dividends,Stock Splits,Returns,MA5,MA20,Volume_MA5,RSI,ATR
Date,,,,,,,,,,,,,
2024-01-02 00:00:00-05:00,4745.200195,4754.330078,4722.669922,4742.830078,3743050000,0.0,0.0,-0.005661,4770.468066,4692.461499,2.966066e+09,64.078086,37.517246
2024-01-03 00:00:00-05:00,4725.069824,4729.290039,4699.709961,4704.810059,3950760000,0.0,0.0,-0.008016,4756.480078,4699.213013,3.253436e+09,56.628431,37.917451
2024-01-04 00:00:00-05:00,4697.419922,4726.779785,4687.529785,4688.680176,3715480000,0.0,0.0,-0.003428,4737.900098,4705.288013,3.446842e+09,53.772233,38.012633
2024-01-05 00:00:00-05:00,4690.569824,4721.490234,4682.109863,4697.240234,3844370000,0.0,0.0,0.001826,4720.678125,4712.683032,3.675944e+09,55.067457,38.110329
2024-01-08 00:00:00-05:00,4703.700195,4764.540039,4699.819824,4763.540039,3742320000,0.0,0.0,0.014115,4719.420117,4721.580542,3.799196e+09,63.579093,40.195291
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-01-14 00:00:00-05:00,5859.270020,5871.919922,5805.419922,5842.910156,4142280000,0.0,0.0,0.001146,5866.690039,5938.841479,4.454896e+09,40.877261,73.992602
2025-01-15 00:00:00-05:00,5905.209961,5960.609863,5905.209961,5949.910156,4544570000,0.0,0.0,0.018313,5874.866113,5933.782495,4.460344e+09,51.391689,77.114538
2025-01-16 00:00:00-05:00,5963.609863,5964.689941,5930.720215,5937.339844,4285810000,0.0,0.0,-0.002113,5878.684082,5926.945483,4.429158e+09,50.260831,74.032766
